# Stage 2 Notebook 56 - Exp2AAA Query K=64 + DAB + DN + VFL + 30 epochs

**Let the DN-DETR queries converge.** NB53 (Exp2XX) showed the highest cls discrimination ever measured (val_lane_best_f1=0.476 at epoch 7, gap=0.157 at epoch 1) but matched_iou stayed at 0.195 and val_det collapsed to 3.35 at epoch 7+. DN-DETR papers (Li et al. 2022) typically need 50+ epochs at standard batch sizes for queries to ground themselves -- 20 epochs at our setup was premature.

Exp2AAA: same head as NB53, but 30 epochs at limit=3000 with cosine LR, lambda_det=1.5 and use_uncertainty=False to prevent the late-training det collapse NB53 showed.

Diffs vs NB53 (Exp2XX exp48):
- `end_epoch: 20 -> 30`
- `lambda_det: 1.0 -> 1.5`, `use_uncertainty: true -> false`
- `warmup_epochs: 2 -> 3`

### Run mode
1. `DEBUG_MODE=True` smoke.
2. `DEBUG_MODE=False` 30 epochs limit=3000. ~50 min.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp51_rmt_gca_query64_dn_vfl_long30_joint_smoke.log
OK exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml
  lane_shape=(1, 64, 72, 2) det_shape=(1, 4, 4)
  lane_loss=3.6368 det_loss=3.4097 grad_cos=0.3872 lambda_lane=0.1376
  gate_stats={'gate/det_mean': 0.5013275146484375, 'gate/lane_mean': 0.49820676445961, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short30'
    EPOCHS = 30
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 50

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]
if LIMIT_TRAIN is not None:
    cmd.extend(['--limit-train', str(LIMIT_TRAIN)])

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('LIMIT_TRAIN:', LIMIT_TRAIN, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
LIMIT_TRAIN: 3000
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp51_rmt_gca_query64_dn_vfl_long30_joint_short30 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp51_rmt_gca_query64_dn_vfl_long30_joint_short30.tar --epochs 30 --batch-size 8 --limit-val 1000 --force-extract --print-every 50 --limit-train 3000
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp51_rmt_gca_query64_dn_vfl_long30_joint_short30.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp51_rmt_gca_query64_dn_vfl_long30_joint_short30_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp51_rmt_gca_query64_dn_vfl_long30_joint.yaml --curve-t

0

## What to watch in Exp2AAA

Reference NB53: matched_iou=0.195 (epoch 20), val_lane_best_f1=0.476 (epoch 7, peak), val_det crashed to 3.3 at epoch 7+.

Pass criteria at epoch 30:
- `val/matched_line_iou >= 0.30` -- 1.5x NB53; longer training should ground queries geometrically.
- `val/lane_best_f1 >= 0.40` -- maintain NB53 peak.
- `val/lane/decoded_f1 >= 0.05` -- 2x NB53.
- `val_det <= 2.5` AT EPOCH 30 -- explicit det rescue should prevent the NB53-style late collapse.